In [1]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import loompy
import velocyto as vcy
import logging
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.interpolate import interp1d

logging.basicConfig(stream=sys.stdout, format='%(asctime)s - %(levelname)s - %(message)s', level=logging.DEBUG)
%matplotlib inline
plt.rcParams['pdf.fonttype'] = 42

2025-05-18 17:51:14,942 - DEBUG - Loaded backend module://matplotlib_inline.backend_inline version unknown.
2025-05-18 17:51:14,943 - DEBUG - Loaded backend inline version unknown.


In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import velocyto as vcy
vlm = vcy.VelocytoLoom("/home/rstudio/run070/run070.loom")

2025-05-18 17:51:14,954 - DEBUG - Creating converter from 3 to 5


AttributeError: `np.string_` was removed in the NumPy 2.0 release. Use `np.bytes_` instead.

In [ ]:
vlm.plot_fractions()

In [ ]:
import numpy as np
vlm.filter_cells(bool_array=vlm.initial_Ucell_size > np.percentile(vlm.initial_Ucell_size, 0.5))

In [ ]:
plt.scatter(vlm.initial_cell_size, vlm.initial_Ucell_size, alpha=0.5, s=5)
plt.axvline(2000, c="r", lw=1)
plt.axvline(np.percentile(vlm.initial_cell_size, 8), c="k", lw=1)
plt.axhline(300, c="r", lw=1)
plt.axhline(np.percentile(vlm.initial_Ucell_size, 8), c="k", lw=1)
plt.xlabel("spliced"); plt.ylabel("unspliced")

In [ ]:
# based on the red line
vlm.filter_cells(bool_array=(vlm.initial_Ucell_size > 300) & (vlm.initial_cell_size > 2000))

In [5]:
import scanpy as sc
import scvelo as scv

ModuleNotFoundError: No module named 'scanpy'

In [ ]:
scv.set_figure_params()

In [ ]:
adata = sc.read_loom('/home/rstudio/run070/run070.loom')

In [ ]:
adata

In [ ]:
# Genes
print(adata.var_names)

# Cells
print(adata.obs_names)

In [ ]:
# All available annotations
#print(adata.layers.head())
print(adata.var.head())

In [ ]:
spliced = adata.layers['spliced']        # Spliced counts
unspliced = adata.layers['unspliced']    # Unspliced counts
ambiguous = adata.layers['ambiguous']    # Ambiguous counts
matrix = adata.layers['matrix']          # Main or total counts

In [ ]:
matrix.shape

In [ ]:
#vlm.ca

In [ ]:
print(adata.layers.keys())

# scVelo

In [4]:
scv.settings.verbosity = 0

NameError: name 'scv' is not defined

In [ ]:
scv.pp.filter_and_normalize(adata)

In [ ]:
scv.pp.moments(adata)

In [ ]:
scv.pl.proportions(adata)

In [ ]:
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

In [ ]:
scv.tl.velocity(adata, mode='stochastic')

In [ ]:
scv.tl.velocity_graph(adata)

In [ ]:
sc.tl.umap(adata)

In [ ]:
scv.pl.velocity_embedding(adata, basis='umap')
scv.pl.velocity_embedding_grid(adata, basis='umap')
scv.pl.velocity_embedding_stream(adata, basis='umap')

In [ ]:
import pandas as pd


df = pd.read_csv("seurat_clusters_patient3.csv")  # Should have all cluster assignments
cellid_to_cluster = dict(zip(df['CellID'], df['Cluster']))  # All clusters, all cells assigned in Seurat

# Now build the full list for adata (length 5755)
ordered_cluster_labels = [cellid_to_cluster.get(cid, 'Unknown') for cid in adata.obs_names]

adata.obs['ClusterName'] = ordered_cluster_labels

In [ ]:
adata.obs['ClusterName'] = adata.obs['ClusterName'].astype('category')

In [ ]:
scv.pl.velocity_embedding_stream(
    adata,
    basis='umap',
    color='ClusterName'   # Use the name of your cluster column
)

In [ ]:
print(adata.obs['ClusterName'].value_counts())

In [ ]:
#Seurat
print(df['CellID'].head(10))

In [ ]:
#AnnData
print(adata.obs_names[:10])

In [ ]:
# Extract the numeric part from AnnData obs_names to have matching names with seurat cell id
numbers = [x.split(":")[1][:-1] for x in adata.obs_names]
#print(numbers[:10])
print(df['CellID'].astype(str).head(10).tolist()) 

In [ ]:
# Build the mapping: number → cluster
cellid_to_cluster = dict(zip(df['CellID'].astype(str), df['Cluster']))

# Assign cluster labels in AnnData(loom)
adata.obs['ClusterName'] = [cellid_to_cluster.get(num, 'Unknown') for num in numbers]

In [ ]:
# Since Seurat ids are integers and cluster names are to be in strings
adata.obs['ClusterName'] = [str(cellid_to_cluster.get(num, 'Unknown')) for num in numbers]

In [ ]:
clusters = adata.obs['ClusterName'].astype('category').cat.categories

colors = plt.cm.tab20(np.linspace(0, 1, len(clusters)))

adata.uns['ClusterName_colors'] = colors

scv.pl.velocity_embedding_stream(
    adata,
    basis='umap',
    color='ClusterName'
)

In [ ]:
scv.pl.velocity_embedding(adata, basis='umap', color='ClusterName')
scv.pl.velocity_embedding_grid(adata, basis='umap', color='ClusterName')
scv.pl.velocity_embedding_stream(adata, basis='umap', color='ClusterName')

In [ ]:
adata

In [ ]:
adata.var

In [ ]:
# TNFRSF9 - immunity gene
scv.pl.velocity(adata, ['TNFRSF9'])

Grey spliced-unspliced plot tells that TNFRSF9 is being upregulated due to higher unspliced cells
Velocity graph points out that some cells are having positive velocity - green color (accumulated at the right side) which all together says that it is transitioning into an activated immune state. Regions with negative velocity may be downregulating, returning to a resting state.

Now these can be compared to clusters to understand which cells(from the cluster names) are main drivers for this activation. This can then be linked to cell activation, exhaustion, or differentiation. In my case however, cluster names are just numbers.

In [ ]:
# to identify important genes according to clusters present in the object. This adds a new column in the object - rank_velocity_genes
scv.tl.rank_velocity_genes(adata, groupby='ClusterName', min_corr=.3)

df = pd.DataFrame(adata.uns['rank_velocity_genes']['names'])
df.head()

In [ ]:
adata.uns['rank_velocity_genes']['names'][:5]

In [ ]:
# This info can be used to compare between two or more conditions to see directionality like in the docu. example (Ngn3 high EP (yellow) to Pre-endocrine (orange) to Beta (green).)

In [ ]:
# Cell cycle detection
## Cell cycle detected by RNA velocity, is biologically affirmed by cell cycle scores (standardized scores of mean expression levels of phase marker genes).

In [ ]:
scv.tl.score_genes_cell_cycle(adata)
scv.pl.scatter(adata, color_gradients=['S_score', 'G2M_score'], smooth=True, perc=[5, 95])

In [ ]:
adata.obs['phase']

In [ ]:
sc.pl.umap(adata, color='S_score') 

In [ ]:
sc.pl.umap(adata, color='G2M_score') 

In [ ]:
# These regions had low gene exp. for TNF immune gene

In [ ]:
# velocity vector
## speed = length of vector
### coherence (how a velocity vector correlates with its neighboring velocities) - confidence measure

In [ ]:
scv.tl.velocity_confidence(adata)
keys = 'velocity_length', 'velocity_confidence'
scv.pl.scatter(adata, c=keys, cmap='coolwarm', perc=[5, 95])

In [ ]:
# confidence rel. high for most regions
# Cells with higher velocity length are likely in active transition states, cells in blue in mature state
# regions with high score in both (red) can be candidates for start/end/branching point in differentiation

In [ ]:
keys = ['velocity_length', 'velocity_confidence'] # changing to list
df = adata.obs.groupby('ClusterName')[keys].mean().T
#df.style.background_gradient(cmap='coolwarm', axis=1)
df.head()

In [ ]:
# matches the plots

In [ ]:
scv.pl.velocity_graph(adata, basis='umap', color='ClusterName')

In [ ]:
scv.tl.velocity_pseudotime(adata)
scv.pl.scatter(adata, color='velocity_pseudotime', basis='umap')

In [ ]:
# cluster 6 the earliest, cluster 2 & possibly 5 the latest

In [ ]:
gene = 'TNFRSF9'
gene_expr = adata[:, gene].X

# If gene_expr is sparse, convert to dense array
if not isinstance(gene_expr, np.ndarray):
    gene_expr = gene_expr.toarray()

# Flatten to 1D
gene_expr = gene_expr.flatten()

# Filter
expressed_cells = np.where(gene_expr > 0)[0]
print(expressed_cells)

In [ ]:
# choosing 25 as the starting point
x, y = scv.utils.get_cell_transitions(adata, basis='umap', starting_cell=25)
ax = scv.pl.velocity_graph(adata, c='lightgrey', edge_width=.05, show=False)
ax = scv.pl.scatter(adata, x=x, y=y, s=120, c='ascending', cmap='gnuplot', ax=ax)

In [ ]:
# Velocity pseudotime
scv.tl.velocity_pseudotime(adata)
scv.pl.scatter(adata, color='velocity_pseudotime', cmap='gnuplot')

In [ ]:
# as expected from previous res.

In [ ]:
#scv.tl.paga(adata, groups='ClusterName')
#df = scv.get_df(adata, 'paga/transitions_confidence', precision=2).T
#df.style.background_gradient(cmap='Blues').format('{:.2g}')

In [ ]:
adata.obs.columns

In [ ]:
sc.tl.leiden(adata, resolution=1.0)